In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Create datetime range (30 days hourly)
date_range = pd.date_range(start='2025-01-01', end='2025-01-30 23:00', freq='H')

# FOOT TRAFFIC
foot_traffic = pd.DataFrame({
    'datetime': date_range,
    'customers_in_store': np.random.poisson(lam=50, size=len(date_range))
})

# Add peak hours effect
foot_traffic['hour'] = foot_traffic['datetime'].dt.hour
foot_traffic['customers_in_store'] += foot_traffic['hour'].apply(
    lambda x: 40 if 17 <= x <= 20 else (20 if 11 <= x <= 13 else 0)
)

# STAFFING (not perfectly aligned → intentional problem)
staffing = pd.DataFrame({
    'datetime': date_range,
    'staff_count': np.random.randint(5, 15, size=len(date_range))
})

# OPERATIONS
operations = pd.DataFrame({
    'datetime': date_range,
    'trolley_collection_time': np.random.normal(loc=10, scale=3, size=len(date_range)).clip(5, 30),
    'cleaning_time': np.random.normal(loc=15, scale=5, size=len(date_range)).clip(5, 40)
})

# TRANSACTIONS (simulate multiple per hour)
transactions = []
for dt in date_range:
    num_transactions = np.random.poisson(lam=30)
    for _ in range(num_transactions):
        transactions.append({
            'datetime': dt,
            'total_amount': np.random.uniform(5, 100),
            'items_count': np.random.randint(1, 20)
        })

transactions = pd.DataFrame(transactions)

# Save files
foot_traffic.to_csv('foot_traffic.csv', index=False)
staffing.to_csv('staffing.csv', index=False)
operations.to_csv('operations.csv', index=False)
transactions.to_csv('transactions.csv', index=False)

In [2]:
import sqlite3
import pandas as pd

# Connect DB
conn = sqlite3.connect('woolworths.db')

# Load CSVs
foot_traffic = pd.read_csv('foot_traffic.csv')
staffing = pd.read_csv('staffing.csv')
operations = pd.read_csv('operations.csv')
transactions = pd.read_csv('transactions.csv')

# Convert datetime
foot_traffic['datetime'] = pd.to_datetime(foot_traffic['datetime'])
staffing['datetime'] = pd.to_datetime(staffing['datetime'])
operations['datetime'] = pd.to_datetime(operations['datetime'])
transactions['datetime'] = pd.to_datetime(transactions['datetime'])

# Load into SQL
foot_traffic.to_sql('foot_traffic', conn, if_exists='replace', index=False)
staffing.to_sql('staffing', conn, if_exists='replace', index=False)
operations.to_sql('operations', conn, if_exists='replace', index=False)
transactions.to_sql('transactions', conn, if_exists='replace', index=False)

21555